In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:
def _add_generation_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from DISPATCH_UNIT_SCADA coal generation (coal_mw).
    A sudden drop in coal MW signals a generator trip — the primary cause
    of extreme NSW price spikes.
    No-op (returns df unchanged) if coal_mw is absent or all-NaN.
    """
    df = df.copy()
    if "coal_mw" not in df.columns or df["coal_mw"].isna().all():
        return df

    coal = df["coal_mw"]

    # Absolute level lags
    for lag in [1, 2, 4, 48, 96, 336]:
        df[f"coal_lag_{lag}"] = coal.shift(lag).astype(np.float32)

    # Rolling statistics
    df["coal_rmean_4"]   = coal.rolling(4).mean().astype(np.float32)
    df["coal_rmean_48"]  = coal.rolling(48).mean().astype(np.float32)
    df["coal_rmin_48"]   = coal.rolling(48).min().astype(np.float32)
    df["coal_rmin_96"]   = coal.rolling(96).min().astype(np.float32)

    # Rate-of-change — rapid drops signal generator trips
    df["coal_chg_1h"]  = coal.diff(2).astype(np.float32)    # 2×30-min = 1h
    df["coal_chg_6h"]  = coal.diff(12).astype(np.float32)   # 6h
    df["coal_chg_24h"] = coal.diff(48).astype(np.float32)   # 24h

    # Outage flag: coal dropped by >600 MW (approx 1 unit) within last 2h
    df["coal_outage_flag"] = (
        coal.rolling(4).min() < coal.shift(4) - 600
    ).astype(np.float32)

    # Capacity utilisation relative to approximate installed max
    df["coal_util"]          = (coal / NSW_COAL_MAX_MW).clip(0, 1.05).astype(np.float32)
    df["coal_low_flag"]      = (coal < NSW_COAL_MAX_MW * 0.55).astype(np.float32)
    df["coal_low_count_48"]  = df["coal_low_flag"].rolling(48).sum().astype(np.float32)
    df["coal_low_count_336"] = df["coal_low_flag"].rolling(336).sum().astype(np.float32)

    return df

In [ ]:
%reset -f